# Cloud Training Runner — Kaggle / Colab (unified 192-D JEPA)

Thin runner around `scripts/train/train_unified.py` + `scripts/eval/eval_scenarios.py`.
Full step-by-step context lives in `CLOUD_TRAINING.md` at the repo root — read that first.
This notebook contains no training logic itself (that lives in the CLI scripts so the same
command works identically on Kaggle, Colab, or locally).

## Mandatory preflight (real data, end-to-end)
```python
!python scripts/train/train_unified.py --config configs/unified.yaml --device cuda --preflight
```
Non-zero exit = DO NOT START TRAINING.

In [ ]:
# ---- CONFIGURE THIS CELL ----
PLATFORM = "kaggle"          # "kaggle" or "colab"
REPO_URL = "https://github.com/<you>/<repo>.git"
REPO_REF = "work-192d"       # branch/tag/commit to train from — pin it deliberately
KAGGLE_DATASET_SLUG = "<your-kaggle-username>/<your-dataset-name>"   # only used if PLATFORM == "kaggle"
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/metasurface-jepa"        # only used if PLATFORM == "colab"
RESUME_FROM = None            # e.g. "checkpoints/unified/latest.pt", or None for a fresh run
VERIFY_MAX_STEPS = 150        # verification-scale run: total_steps=1500 -> 150 steps = 10%
GIT_USER_EMAIL = "you@example.com"
GIT_USER_NAME = "you"


In [ ]:
# ---- GPU CHECK ----
!nvidia-smi

In [ ]:
# ---- MOUNT / ATTACH PERSISTENT STORAGE ----
if PLATFORM == "colab":
    from google.colab import drive
    drive.mount('/content/drive')
    import os
    os.makedirs(f"{DRIVE_PROJECT_DIR}/checkpoints", exist_ok=True)
    os.makedirs(f"{DRIVE_PROJECT_DIR}/data/metadit", exist_ok=True)
elif PLATFORM == "kaggle":
    # Attach the dataset via the Kaggle UI (Add Data) before running this cell.
    !ls /kaggle/input/
else:
    raise ValueError("PLATFORM must be 'kaggle' or 'colab'")

In [ ]:
# ---- CLONE REPO + INSTALL DEPS ----
!git clone {REPO_URL} repo
%cd repo
!git checkout {REPO_REF}
!pip install -r requirements.txt -q

# Provenance (record in checkpoints/unified/REPORT.md)
import sys, subprocess
import torch, torchvision
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"Torchvision: {torchvision.__version__}")
print(f"CUDA: {torch.version.cuda}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU count: {torch.cuda.device_count()}")
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
dirty = subprocess.check_output(['git', 'status', '--porcelain'], text=True).strip()
print(f"Git commit: {commit}")
print(f"Git dirty: {bool(dirty)}")
if dirty:
    print(f"Dirty files:\n{dirty}")

In [ ]:
# ---- LINK DATA + CHECKPOINTS TO PERSISTENT STORAGE ----
if PLATFORM == "kaggle":
    from pathlib import Path
    def discover_metadit_root():
        candidates = []
        for base in [Path("/kaggle/input"), Path("/kaggle/working")]:
            if not base.exists():
                continue
            for p in base.rglob("train_set.mat"):
                root = p.parent.parent
                required = [
                    root / "split_data" / "train_set.mat",
                    root / "split_data" / "val_set.mat",
                    root / "weights" / "spec_encoder.pth",
                ]
                if all(x.exists() for x in required):
                    candidates.append(root)
        candidates = list(dict.fromkeys(str(x.resolve()) for x in candidates))
        if len(candidates) != 1:
            raise RuntimeError(
                f"Expected exactly one valid MetaDiT dataset, found: {candidates}"
            )
        return Path(candidates[0])
    data_root = discover_metadit_root()
    repo_data = Path("data/metadit")
    if repo_data.exists() or repo_data.is_symlink():
        if repo_data.is_symlink() or repo_data.is_file():
            repo_data.unlink()
        else:
            import shutil
            shutil.rmtree(repo_data)
    repo_data.parent.mkdir(parents=True, exist_ok=True)
    repo_data.symlink_to(data_root, target_is_directory=True)
    print("MetaDiT data:", data_root)
    print("surrogate present:", (data_root / "weights" / "surrogate_model.bin").exists())
    # /kaggle/working persists for the session and can be snapshotted as a notebook version
    !mkdir -p /kaggle/working/checkpoints
    !ln -sfn /kaggle/working/checkpoints checkpoints
elif PLATFORM == "colab":
    !ln -sfn {DRIVE_PROJECT_DIR}/data/metadit data/metadit
    !ln -sfn {DRIVE_PROJECT_DIR}/checkpoints checkpoints

In [ ]:
# ---- MANDATORY PREFLIGHT (real data, end-to-end) ----
import subprocess
preflight_cmd = "python scripts/train/train_unified.py --config configs/unified.yaml --device cuda --preflight"
print("Running mandatory preflight...")
result = subprocess.run(preflight_cmd, shell=True)
if result.returncode != 0:
    raise RuntimeError("PREFLIGHT FAILED (exit " + str(result.returncode) + "). DO NOT START TRAINING.")
print("Preflight PASSED.")

In [ ]:
# ---- VERIFICATION-SCALE RUN (5-10% of the schedule) ----
# Proves the full real-data pipeline (data -> step -> EMA -> checkpoint) before
# committing GPU-hours. Checkpoints written here are verification artifacts, NOT results.
!python scripts/train/train_unified.py --config configs/unified.yaml --device cuda --max-steps {VERIFY_MAX_STEPS}
# The shortened schedule affects the LR/EMA ramp: delete the artifacts before the full run
# (alternative: deliberately resume from them).
!rm -f checkpoints/unified/*.pt

In [ ]:
# ---- FULL TRAINING RUN ----
resume_flag = f"--resume {RESUME_FROM}" if RESUME_FROM else ""
!python scripts/train/train_unified.py --config configs/unified.yaml --device cuda {resume_flag}

In [ ]:
# ---- EVALUATION (after training; never pooled across scenarios) ----
CHECKPOINT = RESUME_FROM or "checkpoints/unified/latest.pt"
!python scripts/eval/eval_scenarios.py --config configs/unified.yaml --checkpoint {CHECKPOINT} --device cuda
!python scripts/diagnostics/run_guidance_gap_sweep.py --config configs/unified.yaml --checkpoint {CHECKPOINT} --device cuda

## Sync-back checklist (see `CLOUD_TRAINING.md` §3)

- [ ] `checkpoints/unified/latest.pt` (and `final.pt` if complete) persisted durably
      (Kaggle output / Drive / snapshot) — not only the ephemeral session disk.
- [ ] `checkpoints/unified/REPORT.md` updated: platform/GPU, commit, steps run, losses,
      whether the verification-scale run passed, resume path, deviations.
- [ ] Results pulled back locally before the next coding session.

Acceptance gate reminder: the **hard stratum** (scenario A: full occupancy mask + all scalars
unknown) real-vs-shuffled physics-consistency comparison — never a pooled gap.

In [ ]:
# ---- OPTIONAL: PUSH THE RUN REPORT BACK TO GITHUB ----
# Requires a GitHub personal access token (Kaggle Secret / Colab userdata).
# Checkpoints stay on Kaggle output/Drive — do NOT commit *.pt files.
#
# !git config user.email "{GIT_USER_EMAIL}"
# !git config user.name "{GIT_USER_NAME}"
# !git add checkpoints/unified/REPORT.md
# !git commit -m "unified: cloud training run, see REPORT.md"
# !git push